# Pipeline DNC + Direcciones + Cartografía

Notebook limpio para construir la base documental final:

1. Cargar Datos Abiertos DNC sin headers.
2. Preparar dimensiones y documentos de padrones.
3. Cargar direcciones geográficas solo para la partición necesaria.
4. Cruzar direcciones con padrones mediante join espacial.
5. Exportar JSONL particionado por departamento y tipo de padrón.


In [52]:
# Ejecutar solo si aparece un error de NumPy / pyarrow / pandas.
# Despues de esta celda: reiniciar el kernel y correr el notebook de nuevo.
# %pip install --force-reinstall "numpy<2" "pandas<3" "pyarrow<16"

# Ejecutar solo si falta GeoPandas.
# Despues de esta celda: reiniciar el kernel y correr el notebook de nuevo.
# %pip install geopandas pyogrio


In [53]:
from pathlib import Path
import importlib
import json
import unicodedata

import pandas as pd

DATA_DIR = Path("Data/DatosAbiertosDNC(2026-05)")
DIRECCIONES_DIR = Path("Data/direccionesDepartamentos")
SHAPES_DIR = Path("Data/shapes")
OUTPUT_DIR = Path("salidas")

for path in [DATA_DIR, DIRECCIONES_DIR, SHAPES_DIR]:
    assert path.exists(), f"No existe la carpeta: {path.resolve()}"


In [54]:
TABLES = {
    "padrones_rurales": {
        "file": "Padrones Rurales.csv",
        "columns": [
            "codigo_departamento",
            "nro_padron",
            "seccion_catastral",
            "area_predio",
            "valor_catastral_total",
            "valor_para_impuestos",
        ],
    },
    "padrones_urbanos": {
        "file": "Padrones Urbanos.csv",
        "columns": [
            "codigo_regimen",
            "codigo_departamento",
            "codigo_localidad",
            "nro_padron",
            "block_manzana",
            "ep_ss",
            "unidad",
            "area_predio",
            "area_edificada",
            "valor_catastral_terreno",
            "valor_catastral_mejoras",
            "valor_catastral_total",
            "valor_para_impuestos",
            "fecha_ultima_djcu",
            "vigencia_ultima_djcu",
        ],
    },
    "departamentos": {
        "file": "Departamentos.csv",
        "columns": ["codigo_departamento", "denominacion"],
    },
    "localidades": {
        "file": "Localidades.csv",
        "columns": ["codigo_departamento", "codigo_localidad", "denominacion"],
    },
    "destinos": {
        "file": "Destinos.csv",
        "columns": ["codigo_destino", "denominacion"],
    },
    "categorias_construccion": {
        "file": "Categorias de Construccion.csv",
        "columns": ["codigo_categoria", "denominacion"],
    },
    "estados_conservacion": {
        "file": "Estados de Conservacion.csv",
        "columns": ["codigo_estado", "denominacion"],
    },
    "cubiertas": {
        "file": "Cubiertas.csv",
        "columns": ["codigo_cubierta", "denominacion"],
    },
    "cielorrasos": {
        "file": "Cielorrasos.csv",
        "columns": ["codigo_cielorraso", "denominacion"],
    },
    "tipos_obra": {
        "file": "Tipos de Obra.csv",
        "columns": ["codigo_tipo_obra", "denominacion"],
    },
    "lineas_construccion": {
        "file": "Lineas de Construccion.csv",
        "columns": [
            "codigo_regimen",
            "codigo_departamento",
            "codigo_localidad",
            "nro_padron",
            "block_manzana",
            "ep_ss",
            "unidad",
            "nivel",
            "codigo_destino",
            "categoria_construccion",
            "estado_conservacion",
            "tipo_cubierta",
            "indicador_cielorraso",
            "tipo_obra",
            "area_construida",
            "anio_construccion",
            "anio_remanente",
            "ep_ss_uso_exclusivo",
            "unidad_uso_exclusivo",
        ],
    },
    "mutaciones_catastrales": {
        "file": "Mutaciones Catastrales.CSV",
        "columns": [
            "codigo_regimen",
            "codigo_departamento",
            "seccion_catastral",
            "codigo_localidad",
            "nro_padron",
            "block_manzana",
            "ep_ss",
            "unidad",
            "nro_padron_origen",
            "fecha_vigencia",
            "vigencia_padron_origen",
            "referencia",
        ],
    },
    "historico_valores": {
        "file": "Historico de Valores.CSV",
        "columns": [
            "codigo_regimen",
            "codigo_departamento",
            "codigo_localidad",
            "seccion_catastral",
            "nro_padron",
            "block_manzana",
            "ep_ss",
            "unidad",
            "valor_catastral_anio_1",
            "valor_impuestos_anio_1",
            "valor_catastral_anio_2",
            "valor_impuestos_anio_2",
            "valor_catastral_anio_3",
            "valor_impuestos_anio_3",
            "valor_catastral_anio_4",
            "valor_impuestos_anio_4",
        ],
    },
}

In [55]:
def normalize_key(value: str) -> str:
    value = unicodedata.normalize("NFKD", value)
    value = "".join(ch for ch in value if not unicodedata.combining(ch))
    return value.casefold()


def find_dataset_file(file_name: str) -> Path:
    expected = normalize_key(file_name)
    matches = [path for path in DATA_DIR.iterdir() if normalize_key(path.name) == expected]
    if not matches:
        available = "\n".join(sorted(path.name for path in DATA_DIR.iterdir()))
        raise FileNotFoundError(f"No se encontro {file_name!r}. Archivos disponibles:\n{available}")
    return matches[0]


def read_csv_with_header(table_name: str, spec: dict) -> pd.DataFrame:
    path = find_dataset_file(spec["file"])
    last_error = None

    for encoding in ("utf-8-sig", "latin1"):
        try:
            df = pd.read_csv(
                path,
                header=None,
                sep=",",
                quotechar='"',
                dtype=str,
                keep_default_na=False,
                encoding=encoding,
            )
            break
        except UnicodeDecodeError as exc:
            last_error = exc
    else:
        raise last_error

    expected_columns = spec["columns"]
    if df.shape[1] != len(expected_columns):
        raise ValueError(
            f"{table_name}: el CSV tiene {df.shape[1]} columnas, "
            f"pero el header definido tiene {len(expected_columns)}. Archivo: {path.name}"
        )

    df.columns = expected_columns
    df.attrs["source_file"] = str(path)
    df.attrs["encoding"] = encoding
    return df


dataframes = {
    table_name: read_csv_with_header(table_name, spec)
    for table_name, spec in TABLES.items()
}

globals().update(dataframes)


In [56]:
resumen_dnc = pd.DataFrame(
    [
        {
            "tabla": table_name,
            "archivo": Path(df.attrs["source_file"]).name,
            "encoding": df.attrs["encoding"],
            "filas": len(df),
            "columnas": df.shape[1],
        }
        for table_name, df in dataframes.items()
    ]
).sort_values("tabla")

resumen_dnc


,tabla,archivo,encoding,filas,columnas
5,categorias_construccion,Categorías de Construcción.csv,utf-8-sig,9,2
8,cielorrasos,Cielorrasos.csv,utf-8-sig,1,2
7,cubiertas,Cubiertas.csv,utf-8-sig,5,2
2,departamentos,Departamentos.csv,utf-8-sig,19,2
4,destinos,Destinos.csv,latin1,72,2
6,estados_conservacion,Estados de Conservación.csv,utf-8-sig,9,2
12,historico_valores,Histórico de Valores.CSV,latin1,1668657,16
10,lineas_construccion,Líneas de Construccion.csv,latin1,4478065,19
3,localidades,Localidades.csv,latin1,498,3
11,mutaciones_catastrales,Mutaciones Catastrales.CSV,utf-8-sig,1943,12


## Dimensiones y documentos de padrones

Se reutiliza `base_dnc_documental.py` para armar documentos de padrón con arrays internos de líneas de construcción, histórico de valores y mutaciones.


In [57]:
import base_dnc_documental
importlib.reload(base_dnc_documental)

from base_dnc_documental import (
    clean_record,
    compact_record,
    construir_base_padrones_rurales,
    construir_base_padrones_urbanos,
    preparar_dimensiones,
)


dimensiones_dnc = preparar_dimensiones(
    departamentos=departamentos,
    localidades=localidades,
    destinos=destinos,
    categorias_construccion=categorias_construccion,
    estados_conservacion=estados_conservacion,
    cubiertas=cubiertas,
    cielorrasos=cielorrasos,
    tipos_obra=tipos_obra,
)


## Direcciones geográficas por departamento

Se usan los datos de `Data/direccionesDepartamentos`, descargados de:
https://catalogodatos.gub.uy/dataset/ide-direcciones-geograficas-del-uruguay

Para no cargar todo el país innecesariamente, las funciones leen solo el CSV del departamento solicitado.


In [58]:
def _norm_text(value):
    if pd.isna(value):
        return ""
    value = unicodedata.normalize("NFKD", str(value).strip().upper())
    value = "".join(ch for ch in value if not unicodedata.combining(ch))
    return " ".join(value.split())


def slug_departamento(nombre):
    return _norm_text(nombre).lower().replace(" ", "_")


codigo_departamento_por_nombre = {
    _norm_text(nombre): codigo
    for codigo, nombre in dimensiones_dnc["departamentos"].items()
}

archivo_direcciones_por_codigo_departamento = {
    codigo: DIRECCIONES_DIR / f"{slug_departamento(nombre)}.csv"
    for codigo, nombre in dimensiones_dnc["departamentos"].items()
}

pd.DataFrame(
    [
        {"codigo_departamento": codigo, "archivo_direcciones": str(path), "existe": path.exists()}
        for codigo, path in archivo_direcciones_por_codigo_departamento.items()
    ]
).sort_values("codigo_departamento")


,codigo_departamento,archivo_direcciones,existe
0,A,Data/direccionesDepartamentos/canelones.csv,True
1,B,Data/direccionesDepartamentos/maldonado.csv,True
2,C,Data/direccionesDepartamentos/rocha.csv,True
3,D,Data/direccionesDepartamentos/treinta_y_tres.csv,True
4,E,Data/direccionesDepartamentos/cerro_largo.csv,True
5,F,Data/direccionesDepartamentos/rivera.csv,True
6,G,Data/direccionesDepartamentos/artigas.csv,True
7,H,Data/direccionesDepartamentos/salto.csv,True
8,I,Data/direccionesDepartamentos/paysandu.csv,True
9,J,Data/direccionesDepartamentos/rio_negro.csv,True


In [59]:
def leer_csv_direcciones(path):
    ultimo_error = None
    for encoding in ("utf-8-sig", "latin1"):
        try:
            df = pd.read_csv(
                path,
                dtype=str,
                keep_default_na=False,
                na_values=[],
                encoding=encoding,
            )
            df["archivo_origen"] = path.name
            df["departamento_archivo"] = path.stem
            df.attrs["encoding"] = encoding
            return df
        except UnicodeDecodeError as exc:
            ultimo_error = exc
    raise ultimo_error


def normalizar_para_clave(value):
    if pd.isna(value):
        return ""
    value = str(value).strip()
    if value in {"", "N/A"}:
        return ""
    return _norm_text(value)


def normalizar_coordenada(value, decimales=8):
    if pd.isna(value):
        return pd.NA
    value = str(value).strip().replace(",", ".")
    if value in {"", "N/A"}:
        return pd.NA
    parsed = pd.to_numeric(value, errors="coerce")
    if pd.isna(parsed):
        return pd.NA
    return round(float(parsed), decimales)


def score_completitud_direcciones(df):
    columnas = [
        "codigo_postal", "codigo_via", "nombre_via", "num_puerta", "letra_puerta",
        "km", "manzana", "solar", "nombre_inmueble", "localidad", "codigo_localidad",
        "latitud", "longitud",
    ]
    presentes = df[columnas].replace("N/A", pd.NA).replace("", pd.NA).notna()
    return presentes.sum(axis=1)


def limpiar_duplicados_direcciones(df):
    trabajo = df.copy().drop_duplicates().copy()

    for col in ["departamento", "localidad", "nombre_via", "num_puerta", "letra_puerta"]:
        trabajo[f"{col}_norm"] = trabajo[col].map(normalizar_para_clave)
    trabajo["latitud_norm"] = trabajo["latitud"].map(normalizar_coordenada)
    trabajo["longitud_norm"] = trabajo["longitud"].map(normalizar_coordenada)
    trabajo["score_completitud"] = score_completitud_direcciones(trabajo)
    trabajo["tiene_coord"] = trabajo[["latitud", "longitud"]].replace("N/A", pd.NA).replace("", pd.NA).notna().all(axis=1)

    clave_direccion_norm = [
        "departamento_norm", "localidad_norm", "nombre_via_norm", "num_puerta_norm", "letra_puerta_norm",
    ]

    trabajo = trabajo.sort_values(["score_completitud", "tiene_coord"], ascending=[False, False])
    trabajo = trabajo.loc[~trabajo.duplicated(subset=clave_direccion_norm, keep="first")].copy()

    trabajo["duplicado_coordenada_para_revision"] = trabajo.duplicated(
        subset=["latitud_norm", "longitud_norm"],
        keep=False,
    )

    return trabajo.drop(columns=clave_direccion_norm + ["score_completitud", "tiene_coord"]).reset_index(drop=True)


def leer_direcciones_departamento(codigo_departamento, limpiar=True):
    path = archivo_direcciones_por_codigo_departamento[str(codigo_departamento).strip()]
    if not path.exists():
        raise FileNotFoundError(f"No existe archivo de direcciones para departamento {codigo_departamento}: {path}")
    df = leer_csv_direcciones(path)
    if limpiar:
        df = limpiar_duplicados_direcciones(df)
    df["latitud_num"] = pd.to_numeric(df["latitud"].astype(str).str.strip().str.replace(",", "."), errors="coerce")
    df["longitud_num"] = pd.to_numeric(df["longitud"].astype(str).str.strip().str.replace(",", "."), errors="coerce")
    return df


## Cartografía de padrones

Se usan los shapefiles en `Data/shapes`:

- `PaisUrbano`: polígonos de padrones urbanos.
- `PaisRural`: polígonos de padrones rurales.

La carga es bajo demanda: solo se lee la capa cuando se necesita.


In [60]:
if importlib.util.find_spec("geopandas") is None:
    raise ImportError(
        "Falta geopandas. Ejecutá `%pip install geopandas pyogrio`, reiniciá el kernel y volvé a correr."
    )

import geopandas as gpd

shape_files = sorted(SHAPES_DIR.glob("*/*.shp"))
SHAPE_PATHS = {shp.stem: shp for shp in shape_files}

inventario_shapes = pd.DataFrame(
    [
        {
            "capa": shp.stem,
            "shp": str(shp),
            "tamano_mb_shp": round(shp.stat().st_size / 1024**2, 2),
            "tiene_dbf": shp.with_suffix(".dbf").exists(),
            "tiene_shx": shp.with_suffix(".shx").exists(),
            "tiene_prj": shp.with_suffix(".prj").exists(),
        }
        for shp in shape_files
    ]
)

inventario_shapes


,capa,shp,tamano_mb_shp,tiene_dbf,tiene_shx,tiene_prj
0,PaisLocalidades,Data/shapes/paislocalidades_shp/PaisLocalidade...,0.52,True,True,True
1,PaisRural,Data/shapes/paisrural_shp/PaisRural.shp,226.40,True,True,True
2,PaisSecCat,Data/shapes/paisseccat_shp/PaisSecCat.shp,68.09,True,True,True
3,PaisUrbano,Data/shapes/paisurbano_shp/PaisUrbano.shp,177.59,True,True,True


In [61]:
geodataframes_shapes = {}


def leer_shape(nombre_capa, convertir_a_wgs84=True):
    path = SHAPE_PATHS[nombre_capa]
    kwargs = {}
    if importlib.util.find_spec("pyogrio") is not None:
        kwargs["engine"] = "pyogrio"

    gdf = gpd.read_file(path, **kwargs)
    gdf["capa_origen"] = nombre_capa

    if convertir_a_wgs84 and gdf.crs is not None:
        try:
            gdf = gdf.to_crs(epsg=4326)
        except Exception:
            pass
    return gdf


def cargar_shape_si_falta(nombre):
    if nombre not in geodataframes_shapes:
        geodataframes_shapes[nombre] = leer_shape(nombre)
        globals()[f"shape_{nombre}"] = geodataframes_shapes[nombre]
    return geodataframes_shapes[nombre]


In [62]:
def preparar_claves_padrones_urbanos(df):
    base = df.copy()
    base["CODDEPTO"] = base["codigo_departamento"].astype(str).str.strip()
    base["CODLOCCAT"] = base["codigo_localidad"].astype(str).str.strip()
    base["PADRON"] = pd.to_numeric(base["nro_padron"], errors="coerce").astype("Int64")
    return base


def preparar_claves_padrones_rurales(df):
    base = df.copy()
    base["CODDEPTO"] = base["codigo_departamento"].astype(str).str.strip()
    base["SECCAT"] = pd.to_numeric(base["seccion_catastral"], errors="coerce").astype("Int64")
    base["PADRON"] = pd.to_numeric(base["nro_padron"], errors="coerce").astype("Int64")
    return base


def padrones_urbanos_a_nivel_parcela(df):
    base = preparar_claves_padrones_urbanos(df)
    return (
        base.groupby(["CODDEPTO", "CODLOCCAT", "PADRON"], dropna=False)
        .agg(
            unidades=("unidad", "nunique"),
            regimenes=("codigo_regimen", lambda x: ",".join(sorted(set(x.dropna().astype(str))))),
            area_predio=("area_predio", lambda x: pd.to_numeric(x, errors="coerce").max()),
            area_edificada=("area_edificada", lambda x: pd.to_numeric(x, errors="coerce").sum()),
            valor_catastral_total=("valor_catastral_total", lambda x: pd.to_numeric(x, errors="coerce").max()),
            valor_para_impuestos=("valor_para_impuestos", lambda x: pd.to_numeric(x, errors="coerce").max()),
        )
        .reset_index()
    )


def padrones_rurales_a_nivel_parcela(df):
    base = preparar_claves_padrones_rurales(df)
    return base[[
        "CODDEPTO", "SECCAT", "PADRON", "area_predio", "valor_catastral_total", "valor_para_impuestos"
    ]].copy()


def filtrar_shape_urbano_por_padrones(padrones_seleccionados):
    seleccion = padrones_urbanos_a_nivel_parcela(padrones_seleccionados)
    shape = cargar_shape_si_falta("PaisUrbano").copy()
    shape["CODDEPTO"] = shape["CODDEPTO"].astype(str).str.strip()
    shape["CODLOCCAT"] = shape["CODLOCCAT"].astype(str).str.strip()
    shape["PADRON"] = pd.to_numeric(shape["PADRON"], errors="coerce").astype("Int64")
    return shape.merge(seleccion, on=["CODDEPTO", "CODLOCCAT", "PADRON"], how="inner", suffixes=("_shape", "_dnc"))


def filtrar_shape_rural_por_padrones(padrones_seleccionados):
    seleccion = padrones_rurales_a_nivel_parcela(padrones_seleccionados)
    shape = cargar_shape_si_falta("PaisRural").copy()
    shape["CODDEPTO"] = shape["CODDEPTO"].astype(str).str.strip()
    shape["SECCAT"] = pd.to_numeric(shape["SECCAT"], errors="coerce").astype("Int64")
    shape["PADRON"] = pd.to_numeric(shape["PADRON"], errors="coerce").astype("Int64")
    return shape.merge(seleccion, on=["CODDEPTO", "SECCAT", "PADRON"], how="inner", suffixes=("_shape", "_dnc"))


## Join espacial direcciones ↔ padrones

Se convierte cada dirección en punto geográfico y se asigna al padrón cuyo polígono intersecta ese punto.


In [63]:
def direcciones_a_geodataframe(df_direcciones):
    df = df_direcciones.copy()
    if "latitud_num" not in df.columns:
        df["latitud_num"] = pd.to_numeric(df["latitud"].astype(str).str.strip().str.replace(",", "."), errors="coerce")
    if "longitud_num" not in df.columns:
        df["longitud_num"] = pd.to_numeric(df["longitud"].astype(str).str.strip().str.replace(",", "."), errors="coerce")

    df = df.dropna(subset=["latitud_num", "longitud_num"]).copy()
    return gpd.GeoDataFrame(df, geometry=gpd.points_from_xy(df["longitud_num"], df["latitud_num"]), crs="EPSG:4326")


def direccion_formateada(row):
    partes = []
    nombre_via = row.get("nombre_via")
    num_puerta = row.get("num_puerta")
    letra_puerta = row.get("letra_puerta")
    localidad = row.get("localidad")
    departamento = row.get("departamento")

    if pd.notna(nombre_via) and str(nombre_via).strip() not in {"", "N/A"}:
        calle = str(nombre_via).strip()
        if pd.notna(num_puerta) and str(num_puerta).strip() not in {"", "N/A"}:
            calle += f" {str(num_puerta).strip()}"
        if pd.notna(letra_puerta) and str(letra_puerta).strip() not in {"", "N/A"}:
            calle += f" {str(letra_puerta).strip()}"
        partes.append(calle)
    if pd.notna(localidad) and str(localidad).strip() not in {"", "N/A"}:
        partes.append(str(localidad).strip())
    if pd.notna(departamento) and str(departamento).strip() not in {"", "N/A"}:
        partes.append(str(departamento).strip())
    return ", ".join(partes) if partes else None


def preparar_direcciones_para_join(df_direcciones):
    gdf = direcciones_a_geodataframe(df_direcciones)
    gdf["direccion"] = gdf.apply(direccion_formateada, axis=1)
    gdf["direccion_id"] = gdf.index.astype(str)
    return gdf


def filtrar_direcciones_por_bounds(gdf_direcciones, gdf_poligonos, margen=0.01):
    if gdf_poligonos.empty or gdf_direcciones.empty:
        return gdf_direcciones.iloc[0:0].copy()
    minx, miny, maxx, maxy = gdf_poligonos.total_bounds
    return gdf_direcciones.cx[minx - margen:maxx + margen, miny - margen:maxy + margen].copy()


## Gran base documental de direcciones con padrones

Cada documento representa una dirección geográfica y contiene un array `padrones` con los padrones que matchean espacialmente. Cada padrón mantiene sus arrays internos: `lineas_construccion`, `historico_valores` y `mutaciones`.

La salida se exporta en JSONL particionado por departamento y tipo de padrón.


In [64]:
def direccion_documento(row):
    campos = [
        "latitud", "longitud", "codigo_postal", "codigo_via", "nombre_via", "num_puerta",
        "letra_puerta", "km", "manzana", "solar", "nombre_inmueble", "localidad",
        "codigo_localidad", "departamento", "archivo_origen", "departamento_archivo", "direccion",
    ]
    documento = compact_record({campo: row.get(campo) for campo in campos if campo in row.index})
    documento["latitud_num"] = float(row["latitud_num"]) if pd.notna(row.get("latitud_num")) else None
    documento["longitud_num"] = float(row["longitud_num"]) if pd.notna(row.get("longitud_num")) else None
    return compact_record(documento)


def exportar_documentos_jsonl(documentos, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as file:
        for documento in documentos:
            file.write(json.dumps(documento, ensure_ascii=False, default=str) + "\n")
    return path


def _key_urbano_from_row(row):
    return (
        str(row.get("CODDEPTO", row.get("codigo_departamento"))).strip(),
        str(row.get("CODLOCCAT", row.get("codigo_localidad"))).strip(),
        int(row.get("PADRON", row.get("nro_padron"))),
    )


def _key_rural_from_row(row):
    return (
        str(row.get("CODDEPTO", row.get("codigo_departamento"))).strip(),
        int(row.get("SECCAT", row.get("seccion_catastral"))),
        int(row.get("PADRON", row.get("nro_padron"))),
    )


In [65]:
def lookup_documentos_padrones_urbanos(padrones_seleccionados):
    documentos = construir_base_padrones_urbanos(
        padrones_urbanos=padrones_seleccionados,
        lineas_construccion=lineas_construccion,
        historico_valores=historico_valores,
        mutaciones_catastrales=mutaciones_catastrales,
        dims=dimensiones_dnc,
        limite=None,
        incluir_lineas=True,
    )
    documentos = preparar_claves_padrones_urbanos(documentos)
    lookup = {}
    for _, row in documentos.iterrows():
        key = _key_urbano_from_row(row)
        doc = row.drop(labels=["CODDEPTO", "CODLOCCAT", "PADRON"], errors="ignore").to_dict()
        lookup.setdefault(key, []).append(compact_record(doc))
    return lookup


def lookup_documentos_padrones_rurales(padrones_seleccionados):
    documentos = construir_base_padrones_rurales(
        padrones_rurales=padrones_seleccionados,
        historico_valores=historico_valores,
        mutaciones_catastrales=mutaciones_catastrales,
        dims=dimensiones_dnc,
        limite=None,
    )
    documentos = preparar_claves_padrones_rurales(documentos)
    lookup = {}
    for _, row in documentos.iterrows():
        key = _key_rural_from_row(row)
        doc = row.drop(labels=["CODDEPTO", "SECCAT", "PADRON"], errors="ignore").to_dict()
        lookup.setdefault(key, []).append(compact_record(doc))
    return lookup


In [66]:
def documentos_direcciones_desde_join(join_detalle, padrones_lookup, tipo_padron, key_builder):
    if join_detalle.empty:
        return []

    documentos = []
    for direccion_idx, grupo in join_detalle.groupby("direccion_id", dropna=False):
        direccion_row = grupo.iloc[0]
        padrones_docs = []
        keys_vistas = set()

        for _, match in grupo.iterrows():
            key = key_builder(match)
            if key in keys_vistas:
                continue
            keys_vistas.add(key)
            padrones_docs.extend(padrones_lookup.get(key, []))

        documento = direccion_documento(direccion_row)
        documento["direccion_id"] = str(direccion_idx)
        documento["tipo_padron_match"] = tipo_padron
        documento["cantidad_padrones"] = len(padrones_docs)
        documento["padrones"] = padrones_docs
        documentos.append(documento)

    return documentos


def _join_direcciones_con_mapa(direcciones_particion, mapa_padrones, key_cols):
    direcciones_geo = preparar_direcciones_para_join(direcciones_particion)
    if mapa_padrones.crs is not None and direcciones_geo.crs != mapa_padrones.crs:
        direcciones_geo = direcciones_geo.to_crs(mapa_padrones.crs)

    direcciones_candidatas = filtrar_direcciones_por_bounds(direcciones_geo, mapa_padrones)
    if direcciones_candidatas.empty or mapa_padrones.empty:
        return direcciones_candidatas, pd.DataFrame()

    join_detalle = gpd.sjoin(
        direcciones_candidatas,
        mapa_padrones[key_cols + [mapa_padrones.geometry.name]],
        how="inner",
        predicate="intersects",
    )
    return direcciones_candidatas, join_detalle


In [67]:
def construir_documentos_direcciones_urbanas_particion(codigo_departamento, limite_padrones=1000):
    padrones_particion = padrones_urbanos[padrones_urbanos["codigo_departamento"].eq(codigo_departamento)].copy()
    if limite_padrones is not None:
        padrones_particion = padrones_particion.head(limite_padrones).copy()

    mapa_padrones = filtrar_shape_urbano_por_padrones(padrones_particion)
    direcciones_particion = leer_direcciones_departamento(codigo_departamento, limpiar=True)
    padrones_lookup = lookup_documentos_padrones_urbanos(padrones_particion)

    key_cols = ["CODDEPTO", "CODLOCCAT", "PADRON"]
    direcciones_candidatas, join_detalle = _join_direcciones_con_mapa(direcciones_particion, mapa_padrones, key_cols)
    documentos = documentos_direcciones_desde_join(join_detalle, padrones_lookup, "urbano", _key_urbano_from_row)
    return documentos, mapa_padrones, direcciones_candidatas, join_detalle


def construir_documentos_direcciones_rurales_particion(codigo_departamento, limite_padrones=1000):
    padrones_particion = padrones_rurales[padrones_rurales["codigo_departamento"].eq(codigo_departamento)].copy()
    if limite_padrones is not None:
        padrones_particion = padrones_particion.head(limite_padrones).copy()

    mapa_padrones = filtrar_shape_rural_por_padrones(padrones_particion)
    direcciones_particion = leer_direcciones_departamento(codigo_departamento, limpiar=True)
    padrones_lookup = lookup_documentos_padrones_rurales(padrones_particion)

    key_cols = ["CODDEPTO", "SECCAT", "PADRON"]
    direcciones_candidatas, join_detalle = _join_direcciones_con_mapa(direcciones_particion, mapa_padrones, key_cols)
    documentos = documentos_direcciones_desde_join(join_detalle, padrones_lookup, "rural", _key_rural_from_row)
    return documentos, mapa_padrones, direcciones_candidatas, join_detalle


In [68]:
def construir_base_direcciones_particionada(
    codigos_departamento,
    limite_padrones_por_departamento=1000,
    incluir_urbano=True,
    incluir_rural=False,
    output_dir="salidas/base_direcciones_particionada",
):
    output_dir = Path(output_dir)
    manifiesto = []

    for codigo_departamento in codigos_departamento:
        if incluir_urbano:
            documentos, mapa, candidatas, join_detalle = construir_documentos_direcciones_urbanas_particion(
                codigo_departamento=codigo_departamento,
                limite_padrones=limite_padrones_por_departamento,
            )
            path = exportar_documentos_jsonl(documentos, output_dir / f"departamento={codigo_departamento}" / "tipo=urbano" / "part-00000.jsonl")
            manifiesto.append({
                "codigo_departamento": codigo_departamento,
                "tipo": "urbano",
                "documentos": len(documentos),
                "padrones_shape": len(mapa),
                "direcciones_candidatas": len(candidatas),
                "matches_espaciales": len(join_detalle),
                "archivo": str(path),
            })

        if incluir_rural:
            documentos, mapa, candidatas, join_detalle = construir_documentos_direcciones_rurales_particion(
                codigo_departamento=codigo_departamento,
                limite_padrones=limite_padrones_por_departamento,
            )
            path = exportar_documentos_jsonl(documentos, output_dir / f"departamento={codigo_departamento}" / "tipo=rural" / "part-00000.jsonl")
            manifiesto.append({
                "codigo_departamento": codigo_departamento,
                "tipo": "rural",
                "documentos": len(documentos),
                "padrones_shape": len(mapa),
                "direcciones_candidatas": len(candidatas),
                "matches_espaciales": len(join_detalle),
                "archivo": str(path),
            })

    manifiesto_df = pd.DataFrame(manifiesto)
    output_dir.mkdir(parents=True, exist_ok=True)
    manifiesto_df.to_csv(output_dir / "manifest.csv", index=False)
    return manifiesto_df


## Ejecución

Por defecto se ejecuta una muestra controlada para Canelones (`A`) y 1000 padrones. Para generar una partición completa, cambiar `LIMITE_PADRONES_DIRECCIONES = None`.


In [69]:
DEPARTAMENTOS_DIRECCIONES = ["A"]
LIMITE_PADRONES_DIRECCIONES = 1000

manifest_base_direcciones = construir_base_direcciones_particionada(
    codigos_departamento=DEPARTAMENTOS_DIRECCIONES,
    limite_padrones_por_departamento=LIMITE_PADRONES_DIRECCIONES,
    incluir_urbano=True,
    incluir_rural=False,
)

manifest_base_direcciones


/opt/anaconda3/lib/python3.11/site-packages/pyogrio/raw.py:200: RuntimeWarning: organizePolygons() received an unexpected geometry.  Either a polygon with interior rings, or a polygon with less than 4 points, or a non-Polygon geometry.  Return arguments as a collection.
  return ogr_read(
/Users/gabo/Desktop/Facultad/Herramientas de software para big data/Obligatorio/base_dnc_documental.py:127: UserWarning: Parsing dates in %Y-%m-%d format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  parsed = pd.to_datetime(value, dayfirst=True, errors="coerce")
/Users/gabo/Desktop/Facultad/Herramientas de software para big data/Obligatorio/base_dnc_documental.py:127: UserWarning: Parsing dates in %Y-%m-%d format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  parsed = pd.to_datetime(value, dayfirst=True, errors="coerce")
/Users/gabo/Desktop/Facultad/Herramientas de software para big data/Ob

,codigo_departamento,tipo,documentos,padrones_shape,direcciones_candidatas,matches_espaciales,archivo
0,A,urbano,1127,1000,5150,1127,salidas/base_direcciones_particionada/departam...


In [70]:
# Leer una muestra de la primera partición generada.
if manifest_base_direcciones.empty or manifest_base_direcciones.iloc[0]["documentos"] == 0:
    muestra_base_direcciones = []
else:
    archivo_muestra_direcciones = Path(manifest_base_direcciones.iloc[0]["archivo"])
    cantidad_muestra = min(3, int(manifest_base_direcciones.iloc[0]["documentos"]))
    with archivo_muestra_direcciones.open(encoding="utf-8") as file:
        muestra_base_direcciones = [json.loads(next(file)) for _ in range(cantidad_muestra)]

muestra_base_direcciones


[{'latitud': '-34.5196772138915',
  'longitud': '-56.2809269555346',
  'codigo_postal': '90000',
  'codigo_via': '16328',
  'nombre_via': 'TREINTA Y TRES',
  'num_puerta': '823',
  'letra_puerta': 'N/A',
  'km': 'N/A',
  'manzana': '71',
  'solar': '10',
  'nombre_inmueble': 'N/A',
  'localidad': 'CANELONES',
  'codigo_localidad': '389',
  'departamento': 'CANELONES',
  'archivo_origen': 'canelones.csv',
  'departamento_archivo': 'canelones',
  'direccion': 'TREINTA Y TRES 823, CANELONES, CANELONES',
  'latitud_num': -34.5196772138915,
  'longitud_num': -56.2809269555346,
  'direccion_id': '100164',
  'tipo_padron_match': 'urbano',
  'cantidad_padrones': 1,
  'padrones': [{'codigo_regimen': 'CO',
    'codigo_departamento': 'A',
    'codigo_localidad': 'AA',
    'nro_padron': 235,
    'unidad': 0,
    'area_predio': 4534,
    'area_edificada': 1683,
    'valor_catastral_terreno': 1617395,
    'valor_catastral_mejoras': 5880518,
    'valor_catastral_total': 7497913,
    'valor_para_impue